# Feature Engineering Notebook

Centralized notebook for the NextBuy feature-engineering pipeline: generation, inspection, and modeling readiness.

## 1) Setup and Imports
This notebook uses the existing project module `feature_engineering.py`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from feature_engineering import FeatureEngineer

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

MASTER_PATH = 'master_dataset.csv'
SAMPLE_OUTPUT_PATH = 'features_dataset_sample.csv'
FULL_OUTPUT_PATH = 'features_dataset.csv'

print('Working directory:', os.getcwd())
print('Master dataset exists:', os.path.exists(MASTER_PATH))

## 2) Load Data
You can run in sample mode first for faster iteration.

In [ ]:
df_master = pd.read_csv(MASTER_PATH)
print('Rows:', f"{len(df_master):,}")
print('Columns:', len(df_master.columns))
display(df_master.head(3))

## 3) Optional: Sample Subset for Faster Runs

In [ ]:
USE_SAMPLE = True
SAMPLE_USERS = 1000

if USE_SAMPLE:
    sample_user_ids = df_master['user_id'].dropna().unique()[:SAMPLE_USERS]
    df_input = df_master[df_master['user_id'].isin(sample_user_ids)].copy()
    output_path = SAMPLE_OUTPUT_PATH
else:
    df_input = df_master.copy()
    output_path = FULL_OUTPUT_PATH

print('Using sample mode:', USE_SAMPLE)
print('Input rows:', f"{len(df_input):,}")
print('Unique users:', f"{df_input['user_id'].nunique():,}")

## 4) Run Feature Engineering
Creates user history, product popularity, recency, and cart-position features.

In [ ]:
engineer = FeatureEngineer(df_input)
df_features = engineer.create_all_features(verbose=True)
engineer.get_feature_summary(df_features)
engineer.save_features(df_features, output_path)

print('Saved to:', output_path)
print('Final shape:', df_features.shape)

## 5) Inspect Engineered Features

In [ ]:
original_cols = [
    'order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id',
    'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order',
    'product_name', 'aisle_id', 'department_id', 'aisle', 'department'
]

new_features = [c for c in df_features.columns if c not in original_cols]
user_features = [c for c in new_features if c.startswith('user_')]
product_features = [c for c in new_features if c.startswith('product_')]
up_features = [c for c in new_features if c.startswith('up_')]
other_features = [c for c in new_features if c not in user_features + product_features + up_features]

print('New features:', len(new_features))
print('User features:', len(user_features))
print('Product features:', len(product_features))
print('User-product features:', len(up_features))
print('Other features:', len(other_features))

display(df_features[new_features[:20]].head(3))

## 6) Quick Modeling Readiness Check
Trains a lightweight baseline model to validate signal quality.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

df_model = df_features.copy()
numeric_cols = df_model.select_dtypes(include=[np.number]).columns
df_model[numeric_cols] = df_model[numeric_cols].fillna(0)

exclude_cols = ['order_id', 'product_id', 'user_id']
candidate_cols = [
    c for c in df_model.columns
    if c not in exclude_cols and np.issubdtype(df_model[c].dtype, np.number)
]

X = df_model[candidate_cols]
y = df_model['reordered'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=80,
    max_depth=10,
    min_samples_split=20,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('ROC AUC:', round(roc_auc_score(y_test, y_prob), 4))
print(classification_report(y_test, y_pred, target_names=['Not Reordered', 'Reordered']))

## 7) Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

display(importance_df.head(20))

plt.figure(figsize=(10, 7))
sns.barplot(data=importance_df.head(20), x='importance', y='feature', palette='viridis')
plt.title('Top 20 Feature Importances')
plt.tight_layout()
plt.show()

## 8) Next Steps
- Switch `USE_SAMPLE = False` to run full production feature generation.
- Persist train/test splits for reproducibility.
- Evaluate gradient boosting models (XGBoost/LightGBM) for higher lift.